## Auth


In [32]:
from dotenv import load_dotenv
from os import environ
import requests
from requests.auth import HTTPBasicAuth

# 1. Load the variables from the .env file
load_dotenv()

# 2. Load credentials
credentials = (environ.get('BRAIN_USERNAME'), environ.get('BRAIN_PASSWORD'))

# DEBUG: Print to ensure it is not reading 'None' or including single quotes
print(f"Debug Username: {credentials[0]}")
# print(f"Debug Password: {credentials[1]}") # Uncomment to verify password temporarily

# 3. Extract username and password from the list
username, password = credentials

# 4. Create a session object
sess = requests.Session()

# 5. Set up basic authentication
sess.auth = HTTPBasicAuth(username, password)

# 6. Send a POST request to the API for authentication
response = sess.post('https://api.worldquantbrain.com/authentication')

# 7. Print response status and content for debugging
print(response.status_code)
print(response.json())

Debug Username: xifan9581123@gmail.com
201
{'user': {'id': 'XZ68865'}, 'token': {'expiry': 14400.0}, 'permissions': ['TUTORIAL']}


## 获取数据集ID为 fundamental6 下的所有数据字段


In [11]:
### Get Datafield like Data Explorer 获取所有满足条件的数据字段及其ID
def get_datafields(
    s,
    searchScope,
    dataset_id: str = '',
    search: str = ''
):
    import pandas as pd
    import time

    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']

    if len(search) == 0:
        url_template = "https://api.worldquantbrain.com/data-fields?" +\
            f"&instrumentType={instrument_type}" +\
            f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50" +\
            "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = "https://api.worldquantbrain.com/data-fields?" +\
            f"&instrumentType={instrument_type}" +\
            f"&region={region}&delay={str(delay)}&universe={universe}&limit=50" +\
            f"&search={search}" +\
            "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])
        time.sleep(3)

    datafields_list_flat = [item for sublist in datafields_list for item in sublist]

    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

# 步骤1：配置搜索范围
searchScope = {'region': 'USA', 'delay': '1', 'universe': 'TOP3000', 'instrumentType': 'EQUITY'}

# 步骤2：拉取fundamental6数据集全部字段信息
fundamental6 = get_datafields(s=sess, searchScope=searchScope, dataset_id='fundamental6')

# 步骤3：筛选仅MATRIX类型（可直接写因子）的字段
fundamental6 = fundamental6[fundamental6['type']=="MATRIX"]

# 步骤4：提取所有字段id，转为numpy数组（本次图片代码）
datafields_list_fundamental6 = fundamental6['id'].values

# 输出查看全部字段ID列表
datafields_list_fundamental6

In [12]:
# 搜索范围配置
searchScope = {'region': 'USA', 'delay': '1', 'universe': 'TOP3000', 'instrumentType': 'EQUITY'}

# 调用函数，拉取fundamental6数据集全部字段信息
fundamental6 = get_datafields(s=sess, searchScope=searchScope, dataset_id='fundamental6')

In [13]:
# 数据筛选：只保留类型为 MATRIX 的字段
fundamental6 = fundamental6[fundamental6['type']=="MATRIX"]

# 展示筛选后数据的前5行
fundamental6.head()

,id,description,dataset,category,subcategory,region,delay,universe,type,dateCoverage,coverage,userCount,alphaCount,themes
0,assets,Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,43932,139739,[]
1,assets_curr,Current Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,3408,14466,[]
2,bookvalue_ps,Book Value Per Share,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,2518,9542,[]
3,capex,Capital Expenditures,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,11417,26589,[]
4,cash,Cash,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,1.0,0.5,2479,11467,[]


In [37]:
fundamental6_frequent = fundamental6[fundamental6['alphaCount']>1000]
print(
    fundamental6_frequent.shape,
    fundamental6.shape
)

(169, 14) (574, 14)


In [38]:
# 步骤4：提取所有字段id，转为numpy数组（本次图片代码）
datafields_list_fundamental6_frq = fundamental6_frequent['id'].values

# 输出查看全部字段ID列表
datafields_list_fundamental6_frq


array(['assets', 'assets_curr', 'bookvalue_ps', 'capex', 'cash',
       'cash_st', 'cashflow', 'cashflow_dividends', 'cashflow_fin',
       'cashflow_invst', 'cashflow_op', 'cogs', 'current_ratio', 'debt',
       'debt_lt', 'debt_st', 'depre_amort', 'ebit', 'ebitda', 'employee',
       'enterprise_value', 'eps', 'equity', 'fnd6_acdo', 'fnd6_acodo',
       'fnd6_acox', 'fnd6_acqgdwl', 'fnd6_adesinda_curcd', 'fnd6_aldo',
       'fnd6_am', 'fnd6_aox', 'fnd6_aqc', 'fnd6_capxv', 'fnd6_ceql',
       'fnd6_ch', 'fnd6_ci', 'fnd6_cicurr', 'fnd6_cik', 'fnd6_ciother',
       'fnd6_cisecgl', 'fnd6_city', 'fnd6_cld2', 'fnd6_cld3', 'fnd6_cld4',
       'fnd6_cld5', 'fnd6_cptmfmq_atq', 'fnd6_cptmfmq_ceqq',
       'fnd6_cptmfmq_dlttq', 'fnd6_cptmfmq_lctq', 'fnd6_cptmfmq_opepsq',
       'fnd6_cptnewqv1300_dlttq', 'fnd6_cptnewqv1300_nopiq',
       'fnd6_cptnewqv1300_oeps12', 'fnd6_cptnewqv1300_oiadpq',
       'fnd6_cshtr', 'fnd6_cshtrq', 'fnd6_currencya_curcd',
       'fnd6_currencyqv1300_curcd', 'fnd6_d

In [39]:
### Get Datafield like Data Explorer 获取所有满足条件的数据字段及其ID
def get_datafields(
    s,
    searchScope,
    dataset_id: str = '',
    search: str = ''
):
    import pandas as pd
    import time

    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']

    if len(search) == 0:
        url_template = "https://api.worldquantbrain.com/data-fields?" +\
            f"&instrumentType={instrument_type}" +\
            f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50" +\
            "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = "https://api.worldquantbrain.com/data-fields?" +\
            f"&instrumentType={instrument_type}" +\
            f"&region={region}&delay={str(delay)}&universe={universe}&limit=50" +\
            f"&search={search}" +\
            "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])
        time.sleep(3)

    datafields_list_flat = [item for sublist in datafields_list for item in sublist]

    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

### Test Sample

- 将datafield替换到Alpha模板（框架）中 👉 `group_rank({datafield}/cap, subindustry),` 批量生成Alpha

In [40]:
alpha_list = []

for datafield in datafields_list_fundamental6_frq:
    print("正在将如下Alpha表达式与setting封装")
    alpha_expression = f'group_rank({datafield}/cap, subindustry)'
    print(alpha_expression)
    simulation_data = {
        'type': 'REGULAR',
        'settings': {
            'instrumentType': 'EQUITY',
            'region': 'USA',
            'universe': 'TOP3000',
            'delay': 1,
            'decay': 0,
            'neutralization': 'SUBINDUSTRY',
            'truncation': 0.08,
            'pasteurization': 'ON',
            'unitHandling': 'VERIFY',
            'nanHandling': 'ON',
            'language': 'FASTEXPR',
            'visualization': False,
        },
        'regular': alpha_expression
    }
    alpha_list.append(simulation_data)
print(f'there are {len(alpha_list)} Alphas to simulate')

正在将如下Alpha表达式与setting封装
group_rank(assets/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(assets_curr/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(bookvalue_ps/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(capex/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cash/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cash_st/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cashflow/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cashflow_dividends/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cashflow_fin/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cashflow_invst/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cashflow_op/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(cogs/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(current_ratio/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(debt/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(debt_lt/cap, subindustry)
正在将如下Alpha表达式与setting封装
group_rank(debt_st/cap, subind

## 将Alpha一个一个发送至服务器进行回测
- 有location意味着出现了进度条

In [ ]:
from time import sleep

for alpha in alpha_list:
    sim_resp = sess.post(
        'https://api.worldquantbrain.com/simulations',
        json=alpha,
    )
    try:
        sim_progress_url = sim_resp.headers['Location']
        while True:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = float(sim_progress_resp.headers.get("Retry-After", 0))
            if retry_after_sec == 0: # simulation done!
                break
            sleep(retry_after_sec)
        alpha_id = sim_progress_resp.json()["alpha"] # the final simulation result
        print(alpha_id)
    except:
        print("no location, sleep for 10 seconds and try next alpha")
        sleep(10)

no location, sleep for 10 seconds and try next alpha


## [NEW] Getting IS Testing Status - One alpha simulation sample

In [34]:
import json
from time import sleep

# Grab just the FIRST alpha from your list to test
test_alpha = alpha_list[11]

print(f"Submitting Alpha: {test_alpha['regular']}")

sim_resp = sess.post(
    'https://api.worldquantbrain.com/simulations',
    json=test_alpha,
)

if 'Location' in sim_resp.headers:
    sim_progress_url = sim_resp.headers['Location']
    print("Simulation started. Waiting for completion...")
    
    # 1. Wait for the simulation to finish
    while True:
        sim_progress_resp = sess.get(sim_progress_url)
        retry_after_sec = float(sim_progress_resp.headers.get("Retry-After", 0))
        
        if retry_after_sec == 0:
            break
            
        sleep(retry_after_sec)
        
    sim_data = sim_progress_resp.json()
    alpha_id = sim_data.get("alpha")
    
    # 2. FETCH THE ACTUAL PERFORMANCE METRICS (The missing step!)
    if alpha_id:
        print(f"✅ Simulation finished! Alpha ID: {alpha_id}")
        print("Fetching IS Testing Status...")
        
        # ---> THIS IS THE NEW API CALL <---
        alpha_resp = sess.get(f'https://api.worldquantbrain.com/alphas/{alpha_id}')
        alpha_data = alpha_resp.json()
        
        is_metrics = alpha_data.get("is", {})
        checks = is_metrics.get("checks",[])
        
        print("\n" + "="*40)
        print("EXTRACTED METRICS:")
        print("="*40)
        
        print(f"IS Sharpe:   {is_metrics.get('sharpe')}")
        print(f"IS Fitness:  {is_metrics.get('fitness')}")
        
        # Turnover is usually returned as a decimal (e.g., 0.0249 = 2.49%)
        turnover = is_metrics.get('turnover')
        if turnover is not None:
            print(f"IS Turnover: {turnover * 100:.2f}%")
        
        print("\n" + "="*40)
        print("IS TESTING STATUS:")
        print("="*40)
        
        passed_count = 0
        failed_count = 0
        failed_reasons =

SyntaxError: invalid syntax (1824646175.py, line 60)